In [1]:
import pandas as pd
import numpy as np
pd.options.display.max_rows = 999

In [2]:
class BuySell:

    def __init__(self, capital):
        self.initial_capital = capital
        self.current_capital = capital
        self.share_amount = 0


    def buyandhold(self, dataframe):
        """

        Args:
            dataframe: (pd.DataFrame) should contains price and directive, date columns

        Returns: (float) profit

        """
        first_price = dataframe.iloc[0].loc['price']
        last_price  = dataframe.iloc[-1].loc['price']

        self.current_capital, self.share_amount = BuySell._buy(self.initial_capital, first_price)
        money, self.share_amount = BuySell._sell(self.share_amount, last_price)
        self.current_capital += money

        profit = self.current_capital - self.initial_capital

        return profit

    @staticmethod
    def _buy(money, price):
        """buys if appliable
        Returns: (float, int) current_money, share_amount"""

        remaining_money = money
        share_amount = 0

        if money > price:
            share_amount = int(money / price)
            remaining_money = money - share_amount * price

        return remaining_money, share_amount

    @staticmethod
    def _sell(share_amount, price):
        """sells all shares :)
        Returns: (float, int) current_money, share_amount"""
        money = share_amount * price
        return money, 0



    def process(self, dataframe):
        """

        Args:
            dataframe: (pd.DataFrame) should contains price and directive, date columns

        Returns: (float) profit

        """
        capital_list = list()
        share_amount_list = list()

        for idx,row in dataframe.iterrows():
            if row['directive'] == 'buy':
                remaining_money, share_amount = BuySell._buy(self.current_capital, row['price'])
                self.current_capital = remaining_money
                self.share_amount += share_amount

            if row['directive'] == 'sell':
                money, share_amount = BuySell._sell(self.share_amount, row['price'])
                self.current_capital += money
                self.share_amount = share_amount

            capital_list.append(self.current_capital)
            share_amount_list.append(self.share_amount)

        dataframe.loc[:,'current_capital'] = capital_list
        dataframe.loc[:,'share_amount'] = share_amount_list
        dataframe.loc[:,'total_capital'] = dataframe['current_capital'] + dataframe['share_amount']*dataframe['price']

        dataframe.loc[:,'profit'] = dataframe['total_capital'] - self.initial_capital


        return dataframe
    
def label_to_directives(row):
    row = row[['pbuy','psell','phold']]
    argmax_idx = np.argmax(row.values)

    if argmax_idx == 0:
        return 'buy'
    if argmax_idx == 1:
        return 'sell'
    return 'hold'

In [3]:
# np.random.seed(42)
# size = 100
# fake_dataframe = pd.DataFrame({'price':np.random.randint(100,150, size=size),
#                                'directive':np.random.choice(['buy','hold','sell'], size=size, replace=True),
#      'name':np.random.choice(['xlp','xlu','xlv','xly','dia','ewa','ewc','ewg','ewh','ewj','eww','spy','xlb','xle','xlf','xli','xlk'], size=size, replace=True)})


In [4]:
initial_capital = 100000
stock_names = ['dia', 'ewa', 'ewc', 'ewg', 'ewh', 'ewj', 'eww', 'spy', 'xlb', 'xle', 'xlf', 'xli', 'xlk', 'xlp', 'xlu', 'xlv', 'xly']
exp_name = 'stock_exp'

final_result_df = None
for stock_name in stock_names:
    path = '../experiment/finance_cnn/{}/{}/prediction_results.csv'.format(exp_name,stock_name)
    dataframe = pd.read_csv(path)
    dataframe['directive'] = dataframe.apply(label_to_directives, axis=1)
    dataframe['price'] = dataframe['raw_adjusted_close'].values
    
    # Buy and Hold strategy
    buyandhold_profit = BuySell(capital=initial_capital).buyandhold(dataframe)
    
    # Our simple strategy
    buysell = BuySell(capital=initial_capital)
    buysell_result_df = buysell.process(dataframe)
    subset = ['name','date','price','directive','current_capital','share_amount','total_capital', 'profit']
#     subset = buysell_result_df.columns
    if final_result_df is None:
        final_result_df = pd.DataFrame(columns=subset)
    
    last_row = buysell_result_df[subset].iloc[-1]
    last_row['bah_profit'] = buyandhold_profit
    
    final_result_df = final_result_df.append(last_row)
    
final_result_df = final_result_df.reset_index(drop=True)


,name,date,price,directive,current_capital,share_amount,total_capital,profit,bah_profit
0,dia,2016-11-21,189.34,hold,32.28,513,97163.70,-2836.30,5531.01
1,ewa,2016-11-18,20.03,hold,127974.11,0,127974.11,27974.11,-8328.32
2,ewc,2016-11-18,25.46,buy,12.18,4834,123085.82,23085.82,-7518.24
3,ewg,2016-11-17,25.09,hold,104499.89,0,104499.89,4499.89,-15117.54
4,ewh,2016-11-17,20.71,hold,6.38,4747,98316.75,-1683.25,-10034.64
5,ewj,2016-11-17,50.14,hold,14.88,2285,114584.78,14584.78,-2828.02
6,eww,2016-11-18,42.97,hold,20.77,1871,80417.64,-19582.36,-26363.07
7,spy,2016-11-18,218.50,hold,100.05,513,112190.55,12190.55,6817.36
8,xlb,2016-11-21,48.89,hold,29.15,2240,109542.75,9542.75,-1470.95
9,xle,2016-11-17,70.84,hold,15.35,1340,94940.95,-5059.05,-8106.25


In [5]:
initial_capital = 100000
exp_name = 'stress_exp'
final_result_df = None
for i in range(19):
    path = '../experiment/finance_cnn/{}/{}/prediction_results.csv'.format(exp_name,i)
    dataframe = pd.read_csv(path)
    dataframe['directive'] = dataframe.apply(label_to_directives, axis=1)
    dataframe['price'] = dataframe['raw_adjusted_close'].values
    
    # Buy and Hold strategy
    buyandhold_profit = BuySell(capital=initial_capital).buyandhold(dataframe)
    
    # Our simple strategy
    buysell = BuySell(capital=initial_capital)
    buysell_result_df = buysell.process(dataframe)
    subset = ['name','date','price','directive','current_capital','share_amount','total_capital', 'profit']
#     subset = buysell_result_df.columns
    if final_result_df is None:
        final_result_df = pd.DataFrame(columns=subset)
    
    last_row = buysell_result_df[subset].iloc[-1]
    last_row['bah_profit'] = buyandhold_profit
    last_row['deleted_colnum'] = i
    
    final_result_df = final_result_df.append(last_row)
    
final_result_df = final_result_df.reset_index(drop=True)

final_result_df

,name,date,price,directive,current_capital,share_amount,total_capital,profit,bah_profit,deleted_colnum
0,dia,2016-11-21,189.34,hold,120430.77,0,120430.77,20430.77,5531.01,0.0
1,dia,2016-11-21,189.34,hold,68.30,563,106666.72,6666.72,4986.00,1.0
2,dia,2016-11-21,189.34,hold,90641.54,0,90641.54,-9358.46,4986.00,2.0
3,dia,2016-11-21,189.34,hold,63.85,549,104011.51,4011.51,4986.00,3.0
4,dia,2016-11-21,189.34,hold,145.94,532,100874.82,874.82,4986.00,4.0
5,dia,2016-11-21,189.34,buy,87.59,558,105739.31,5739.31,4986.00,5.0
6,dia,2016-11-21,189.34,buy,1.15,554,104895.51,4895.51,4986.00,6.0
7,dia,2016-11-21,189.34,hold,130.78,570,108054.58,8054.58,4986.00,7.0
8,dia,2016-11-21,189.34,hold,16.20,599,113430.86,13430.86,4986.00,8.0
9,dia,2016-11-21,189.34,hold,94.17,559,105935.23,5935.23,4986.00,9.0
